# Anomaly Investigation

Investigate the sales spike (~480M) around late 2019 and the dip (~100M) in early 2019 seen in the monthly trend chart.

In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
from eda_utils import load_clean_data

df = load_clean_data("../data/processed/cleaned_data.csv")
df.head()

## 1. Monthly sales table - find exact peak/dip months

In [ ]:
monthly = df.groupby(pd.Grouper(key='date', freq='ME'))['sales'].sum().reset_index()
# df['date'] is the 1st of each month; 'ME' grouping returns month-end dates.
# Normalize to month-start so peak_month/dip_month match df['date'] for filtering below.
monthly['date'] = monthly['date'].values.astype('datetime64[M]')
monthly = monthly.sort_values('sales')
print("Lowest months:")
print(monthly.head(5))
print("\nHighest months:")
print(monthly.sort_values('sales', ascending=False).head(5))

## 2. Drill into the peak month - row count, distributors, products

In [ ]:
peak_month = monthly.sort_values('sales', ascending=False).iloc[0]['date']
print(f"Peak month: {peak_month}")

peak_df = df[df['date'] == peak_month]
print(f"\nRow count in peak month: {len(peak_df)}")
print(f"Average row count in other months: {len(df) / df['date'].nunique():.0f}")

print("\nTop contributing rows (by sales):")
peak_df.sort_values('sales', ascending=False).head(10)[
    ['distributor', 'customer_name', 'city', 'country', 'product_name', 'quantity', 'price', 'sales']
]

In [ ]:
# Check: is the spike driven by a few huge rows, or broadly higher sales across many rows?
print("Peak month sales stats:")
print(peak_df['sales'].describe())

print("\nOverall sales stats (all months):")
print(df['sales'].describe())

In [ ]:
# Check for duplicate rows specifically in the peak month
print(f"Duplicates in peak month: {peak_df.duplicated(subset=['distributor','customer_name','city','product_name','quantity','price']).sum()}")

# Check distribution by country/channel for peak month vs normal
print("\nPeak month sales by country:")
print(peak_df.groupby('country')['sales'].sum())

print("\nPeak month sales by product class:")
print(peak_df.groupby('product_class')['sales'].sum().sort_values(ascending=False))

## 3. Drill into the dip month

In [ ]:
dip_month = monthly.iloc[0]['date']
print(f"Dip month: {dip_month}")

dip_df = df[df['date'] == dip_month]
print(f"\nRow count in dip month: {len(dip_df)}")
print(f"Average row count in other months: {len(df) / df['date'].nunique():.0f}")

print("\nDip month sales by country:")
print(dip_df.groupby('country')['sales'].sum())

print("\nDip month sales by product class:")
print(dip_df.groupby('product_class')['sales'].sum().sort_values(ascending=False))

## 4. Row counts per month - check for a structural data issue (e.g. missing/extra data dump)

In [ ]:
row_counts = df.groupby(pd.Grouper(key='date', freq='ME')).size().reset_index(name='row_count')
row_counts['date'] = row_counts['date'].values.astype('datetime64[M]')
row_counts_merged = row_counts.merge(monthly, on='date')
row_counts_merged.sort_values('sales')

## 5. Outlier rows overall - top 20 single transactions by sales value

In [ ]:
df.sort_values('sales', ascending=False).head(20)[
    ['date', 'distributor', 'customer_name', 'city', 'country', 'product_name', 'quantity', 'price', 'sales']
]

## 6. Identify the geo hotspot city from the scatter plot

In [ ]:
geo = df.groupby(['city', 'country', 'latitude', 'longitude'])['sales'].sum().reset_index()
geo.sort_values('sales', ascending=False).head(10)

## 7. Negative sales investigation (data quality issue)

In [ ]:
neg = df[df['sales'] < 0]
print(f"Rows with negative sales: {len(neg)} ({len(neg)/len(df)*100:.3f}% of data)")
print(f"Total negative sales value: {neg['sales'].sum():,.0f}")
print("\nNegative sales by country:")
print(neg.groupby('country')['sales'].agg(['count', 'sum']))
print("\nNegative sales by product class:")
print(neg.groupby('product_class')['sales'].agg(['count', 'sum']).sort_values('sum'))
print("\nQuantity stats for negative-sales rows:")
print(neg['quantity'].describe())
print("\nSample negative rows:")
neg.sort_values('sales').head(10)[['date','distributor','customer_name','city','product_name','quantity','price','sales']]

## 8. Bulk vs retail transaction split

Check whether a small number of very large (bulk/wholesale) transactions are driving overall totals - relevant for clustering and forecasting steps.

In [ ]:
# Define a bulk threshold - adjust based on the distribution (e.g. 99th percentile of quantity)
bulk_threshold = df['quantity'].quantile(0.99)
print(f"99th percentile quantity (bulk threshold): {bulk_threshold:.0f}")

df['order_type'] = (df['quantity'] >= bulk_threshold).map({True: 'bulk', False: 'retail'})

order_summary = df.groupby('order_type').agg(
    row_count=('sales', 'count'),
    total_sales=('sales', 'sum')
)
order_summary['pct_of_rows'] = order_summary['row_count'] / order_summary['row_count'].sum() * 100
order_summary['pct_of_sales'] = order_summary['total_sales'] / order_summary['total_sales'].sum() * 100
order_summary

In [ ]:
# Distributors most associated with bulk orders
df[df['order_type'] == 'bulk'].groupby('distributor')['sales'].agg(['count','sum']).sort_values('sum', ascending=False).head(10)

## Conclusions

- **Peak (Aug 2019, ~480M)**: row count (4446) is near average - NOT a data dump. Driven by a handful of very large transactions, mostly from distributor Bashirian-Kassulke, dated 2019-08-01, in Germany. Quantities of 50K-117K units indicate bulk/wholesale orders.
- **Dip (Jan 2019, ~98M)**: also near-average row count - check section 3 output for whether it's a genuine slow month or absence of bulk orders that month.
- **Negative sales**: small % of rows, likely represent returns/credits. Decide: keep as a 'returns' signal, or filter out before forecasting/modeling (recommended for revenue forecasting).
- **Bulk vs retail split**: a small % of rows (top quantity percentile) likely account for a large % of total sales. For Step 3 (segmentation) and Step 5 (forecasting), consider:
  - Adding an `order_type` (bulk/retail) feature
  - Modeling/clustering bulk and retail separately, or capping/winsorizing extreme values for forecasting
- **Geo hotspot (Butzbach, Germany, ~93.5M)**: driven almost entirely by one 74.2M bulk transaction in Aug 2019 - not a sustained high-value territory. Exclude or flag when ranking territories for the SFE story.